In [86]:
# Basics

"""
1. Important Runnables in langchain - Single function, sequenqial, parallel, branching
2. LCEL - custom runnables, control flow, error handling
3. LCEL - invoke llm calls with string output
4. LCEL - invoke llm calls with structured output
5. LCEL - batch calls, event streaming
"""

'\n1. Important Runnables in langchain - Single function, sequenqial, parallel, branching\n2. LCEL - custom runnables, control flow, error handling\n3. LCEL - invoke llm calls with string output\n4. LCEL - invoke llm calls with structured output\n5. LCEL - batch calls, event streaming\n'


#### Why runnables are important?

- **Abstraction** - Runnables abstract away the underlying implementation details of a task, allowing developers to focus on the high-level logic of their applications. This abstraction makes it easier to build complex applications without getting bogged down in the specifics of how each task is executed.

- **Reusability** - Runnables can be reused across different parts of an application or even across different applications.

- **Composability** - Runnables can be composed together to create more complex workflows.


In [51]:
from langchain_core.runnables import RunnableLambda

In [52]:
input_text = {
    "input_text": "test call to the runnable"
}

In [53]:
def uppercase_from_dict(input_dict: dict) -> str:
    return input_dict["input_text"].upper()


In [ ]:
# We start with a simple RunnableLambda which takes a function and makes it runnable. 

runnable_2 = RunnableLambda(uppercase_from_dict)

In [ ]:
# To call any runnable, it has to be invoked with corrent input type.

result = runnable_2.invoke(input_text)
print(result)

TEST CALL TO THE RUNNABLE


In [54]:
# This is where runnables get interesting. We can compose them in sequence, parallel, branching etc.

In [34]:
def function_one(x: str) -> str:
    """First function - takes string, returns modified string"""
    return x.upper()

In [35]:

def function_two(x: str) -> dict:
    """Second function - takes string, returns dict"""
    return {"text": x, "length": len(x)}

In [36]:
def function_three(x: dict) -> dict:
    """Third function - takes dict, returns enriched dict"""
    x["processed"] = True
    x["summary"] = f"Processed text of length {x['length']}"
    return x

In [37]:
# Wrap each function as a runnable
runnable_one = RunnableLambda(function_one)
runnable_two = RunnableLambda(function_two)
runnable_three = RunnableLambda(function_three)

In [38]:
# Chain them sequentially
chain = runnable_one | runnable_two | runnable_three

In [40]:
result = chain.invoke("Run custom functions in sequence")
print(result)

{'text': 'RUN CUSTOM FUNCTIONS IN SEQUENCE', 'length': 32, 'processed': True, 'summary': 'Processed text of length 32'}


In [41]:
from langchain_core.runnables import RunnableParallel

In [ ]:
# When same input has to be passed to multiple functions and run in parallel, we can use RunnableParallel. 
# This also gets useful for non-genAI usecases where we want to run multiple functions in parallel and then aggregate the results.

parallel_chain = RunnableParallel(
    uppercase=runnable_one,
    with_length=runnable_two
)

In [44]:
result = parallel_chain.invoke("Run functions in parallel")
print(result)

{'uppercase': 'RUN FUNCTIONS IN PARALLEL', 'with_length': {'text': 'Run functions in parallel', 'length': 25}}


In [45]:
from langchain_core.runnables import RunnableBranch

In [ ]:
# Define branching logic
# Mostly I have seen this used with conditions based on input type or content, but it can be used for any custom branching logic.

branch = RunnableBranch(
    (lambda x: isinstance(x, dict), runnable_three),  # If dict, use runnable_three
    runnable_two  # Default: use runnable_two
)

In [47]:
result1 = branch.invoke("test string")
print("String input:", result1)

String input: {'text': 'test string', 'length': 11}


In [48]:
result2 = branch.invoke({"text": "TEST", "length": 4})
print("Dict input:", result2)

Dict input: {'text': 'TEST', 'length': 4, 'processed': True, 'summary': 'Processed text of length 4'}


#### LCEL


In [55]:
from langchain_core.prompts import ChatPromptTemplate # This is a good, clean way to create prompts with variables.

In [56]:
from langchain_ollama import ChatOllama # If you want to use ollama as your llm provider
from langchain_openai import ChatOpenAI # If you want to use openai as your llm provider. Similarly, langchain has integrations with many other llm providers.

In [57]:
from langchain_core.output_parsers import StrOutputParser # This is a simple output parser that expects a string output.

In [58]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])

In [59]:
model = ChatOllama(model="mistral:7b")

In [60]:
chain = prompt | model | StrOutputParser()

In [61]:
result = chain.invoke({"question": "What is the capital of India?"})
print(result)

 The capital of India is New Delhi.


In [ ]:
# Now that we have seen how to generate responses from LLMs, sometimes we may want the response to follow a specific format.
# Simple way to structure a format is to give it in system prompt itself. But that does not guarantee that the model will follow the format always.

# This is where I initially had a question - how to ensure that the model follows the output format?
# Here my experience says instead of going for complex prompt engineering to make the model follow the format, it is better to use PYDANTIC models.

In [63]:
from typing import List, Optional, Union, Dict, Any, TypedDict

In [64]:
class ResponseModel(TypedDict):
    city: str
    coordinates: Dict[str, float]

In [65]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant.
     You will receive the quastion where the answer can be name of a place and co-ordinates of that place.
     """),
    ("human", "{question}")
])

In [66]:
structured_model = ChatOllama(model="mistral:7b").with_structured_output(ResponseModel)

In [67]:
structured_chain = prompt | structured_model

In [ ]:
result = structured_chain.invoke({"question": "What is the capital of France?"})
print(result) # Result will be a dict with 'city' and 'coordinates' keys, and the model will be forced to follow this format.

{'city': 'Paris', 'coordinates': {'latitude': 48.864716, 'longitude': 2.349014}}


In [70]:
# My personal preference is to use structured output with pydantic models, as it gives more control and guarantees on the output format, rather than relying on prompt engineering to make the model follow the format.

In [ ]:
# Another useful feature of langchain is how easily it allows batch and streaming calls.

In [77]:
batch_result = structured_chain.batch([
    {"question": "Which place is considered the birthplace of Buddha?"},
    {"question": "What is the most ancient city in the world?"},
    {"question": "Where is ISRO located?"}
])

In [78]:
print(batch_result)

[{'city': 'Lumbini', 'coordinates': {'latitude': 26.818, 'longitude': 83.2397}}, {'city': 'Jericho', 'coordinates': {'latitude': 31.9872, 'longitude': 35.4628}}, {'city': 'Bangalore', 'coordinates': {'latitude': 12.9667, 'longitude': 77.5833}}]


In [79]:
async_batch_result = await structured_chain.abatch([
    {"question": "Where was the first olympics played?"},
    {"question": "What is the least crowded city in the world?"},
    {"question": "Where is IMA located?"}
])

In [80]:
print(async_batch_result)

[{'city': 'Athens', 'coordinates': {'latitude': 37.98381, 'longitude': 23.72754}}, {'city': 'Numanna, Myanmar (21.8333° N, 95.7667° E)', 'coordinates': {'latitude': 21.8333, 'longitude': 95.7667}}, {'city': 'Bangalore', 'coordinates': {'latitude': 12.9667, 'longitude': 77.5833}}]


In [82]:
# Stream the responses

for chunk in structured_chain.stream([
        {"question": "What is the capital of Germany?"},
    ]):
    print("Received chunk:", chunk, end="", flush=True)

Received chunk: {}Received chunk: {'city': ''}Received chunk: {'city': 'Ber'}Received chunk: {'city': 'Berlin'}Received chunk: {'city': 'Berlin', 'coordinates': {}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 5}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.5}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.52}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.520008}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.520008, 'longitude': 1}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.520008, 'longitude': 13}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.520008, 'longitude': 13.4}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.520008, 'longitude': 13.405}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.520008, 'longitude': 13.405004}}

In [85]:
# Async streaming of the responses

async for chunk in structured_chain.astream([
        {"question": "What is the capital of Germany?"},
    ]):
    
    print("Received chunk:", chunk, end="", flush=True)

Received chunk: {}Received chunk: {'city': ''}Received chunk: {'city': 'Ber'}Received chunk: {'city': 'Berlin'}Received chunk: {'city': 'Berlin', 'coordinates': {}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 5}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.5}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.52}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.52, 'longitude': 1}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.52, 'longitude': 13}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.52, 'longitude': 13.4}}Received chunk: {'city': 'Berlin', 'coordinates': {'latitude': 52.52, 'longitude': 13.405}}

In [ ]:
# Until this point, we have seen ho to generate responses how we want - with custom formats, in batch, and even stream the responses. This is all possible with the way we have structured our chain. We can mix and match different runnables, prompts, models, output parsers etc to create the desired flow and output.
# This is the point where first-timers should start with chatbot development.
# Refer to this branch next : beginners-app